# Menu Targeting — User Segments Export 14_09_2026
將 8 個 targeting group 嘅 UserID 寫出 Excel，每個 Group 一個 Sheet。

| Sheet Name | Group | Promo |
|------------|-------|-------|
| Sleeping User (A) | Gp1 — 有買 menu Sep 2025-Feb 2026，但 Mar-Aug 2026 冇買 | $400-30 |
| Warm User | Gp2 — Aug 2026 有買 menu | $400-30 |
| NEW User (of previous mos) | Gp3 — 第一次買 menu 係 Jul 2026，Aug 2026 冇買 | $400-30 |
| Booking Standing 5 Star | Gp4 | $800-60 |
| Sleeping User (B) | Gp5 — 有買 menu Sep 2024-Aug 2025，但過去 12 個月 (Sep 2025-Aug 2026) 冇買 | $400-30 |
| NEW NEW User (A) | Gp6 — 有 VOU txn 單筆 $200+，但冇買過 menu (Sep 2025-Aug 2026) | $400-30 |
| NEW NEW User (B) | Gp7 — 有 VOU txn 喺 bookable POI，但冇買過 menu (Sep 2025-Aug 2026) | $400-30 |
| Premium User | Gp8 — Sep 2025-Aug 2026 | $1600-120 |

In [15]:
# Install dependencies if needed
# !pip install pymssql pandas openpyxl
import pymssql
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import datetime

print('Libraries loaded OK')

Libraries loaded OK


In [16]:
# ─── DB Connection ────────────────────────────────────────────
DB_SERVER   = '192.168.61.119'
DB_PORT     = 7622
DB_USER     = 'BAReporting'
DB_PASSWORD = 'KeHeCReme8he'

OUTPUT_PATH = r'Menu_Targeting_UserSegments.xlsx'

conn = pymssql.connect(
    server=DB_SERVER, port=DB_PORT,
    user=DB_USER, password=DB_PASSWORD,
    database='Mars', charset='UTF-8'
)
print('DB connected')

DB connected


## Step 1 — 建立共用 Temp Tables 及查詢各 Group

In [17]:
# ─── 定義每個 Group 嘅 SQL ────────────────────────────────────
# 每條 SQL 係獨立嘅 (使用 CTE/subquery)，唔依賴 temp table
# 返回欄位：userid
# 所有 query 都加咗 User.status = 10 (active user)

# Sheet 出場順序 (跟參考 Excel)
SHEET_ORDER = ['Gp1', 'Gp2', 'Gp3', 'Gp4', 'Gp5', 'Gp6', 'Gp7', 'Gp8']

groups = {}

# ── Gp1: Sleeping User (A) ──────────────────────────────────
groups['Gp1'] = {
    'sheet_name': 'Sleeping User (A)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        may_oct AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2025-09-01' AND PaymentTime < '2026-03-01'
        ),
        nov_apr AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2026-03-01' AND PaymentTime < '2026-09-01'
        )
        SELECT DISTINCT g.userid
        FROM may_oct g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.userid NOT IN (SELECT userid FROM nov_apr)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp2: Warm User ───────────────────────────────────────────
groups['Gp2'] = {
    'sheet_name': 'Warm User',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT g.userid
        FROM menu_buyers g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.PaymentTime >= '2026-08-01' AND g.PaymentTime < '2026-09-01'
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp3: New User (of previous mos) ──────────────────────────────────────
groups['Gp3'] = {
    'sheet_name': 'New User(of previous mos)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        first_buy AS (
            SELECT userid, MIN(PaymentTime) AS first_buy_time FROM menu_buyers GROUP BY userid
        ),
        apr_buyers AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2026-08-01' AND PaymentTime < '2026-09-01'
        )
        SELECT DISTINCT g.userid
        FROM first_buy g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.first_buy_time >= '2026-07-01' AND g.first_buy_time < '2026-08-01'
          AND g.userid NOT IN (SELECT userid FROM apr_buyers)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp4: Booking Standing 5 Star ────────────────────────────
groups['Gp4'] = {
    'sheet_name': 'Booking Standing 5 Star',
    'promo':   '$800-60',
    'sql': """
        USE Mars;
        SELECT DISTINCT u.userid
        FROM [openrice3].[dbo].[User] ou (NOLOCK)
        INNER JOIN [Mars].[dbo].[user] u (NOLOCK) ON u.SSOUserId = ou.SSOUserId
        WHERE ou.UserStar = 5
          AND u.status = 10
        ORDER BY u.userid
    """
}

# ── Gp5: Sleeping User (B) ──────────────────────────────────
groups['Gp5'] = {
    'sheet_name': 'Sleeping User (B)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH menu_buyers AS (
            SELECT b.userid, bp.PaymentTime
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND b.userid IS NOT NULL AND b.userid != 0
        ),
        prev_year AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2024-09-01' AND PaymentTime < '2025-09-01'
        ),
        last_12m AS (
            SELECT DISTINCT userid FROM menu_buyers
            WHERE PaymentTime >= '2025-09-01' AND PaymentTime < '2026-09-01'
        )
        SELECT DISTINCT g.userid
        FROM prev_year g
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = g.userid
        WHERE g.userid NOT IN (SELECT userid FROM last_12m)
          AND u.status = 10
        ORDER BY g.userid
    """
}

# ── Gp6: NEW NEW User (A) ──────────────────────────────────
groups['Gp6'] = {
    'sheet_name': 'NEW NEW User (A)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH vou_txn AS (
            SELECT u.userid, vo.FinalPrice
            FROM VoucherOrder vo (NOLOCK)
            INNER JOIN [mars].[dbo].[User] u (NOLOCK) ON u.SSOUserId = vo.SSOUserId
            WHERE vo.status = 10
              AND vo.PaymentTime >= '2025-09-01' AND vo.PaymentTime < '2026-09-01'
              AND u.status = 10
        ),
        menu_buyers_12m AS (
            SELECT DISTINCT b.userid
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND bp.PaymentTime >= '2025-09-01' AND bp.PaymentTime < '2026-09-01'
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT g.userid
        FROM vou_txn g
        WHERE g.FinalPrice >= 200
          AND g.userid NOT IN (SELECT userid FROM menu_buyers_12m)
        ORDER BY g.userid
    """
}

# ── Gp7: NEW NEW User (B) ──────────────────────────────────
groups['Gp7'] = {
    'sheet_name': 'NEW NEW User (B)',
    'promo':   '$400-30',
    'sql': """
        USE Mars;
        WITH bookable_poi AS (
            SELECT DISTINCT poiid
            FROM BizService (NOLOCK)
            WHERE ServiceTypeId IN (1, 101)
              AND status = 10
              AND ServiceStartTime <= '2025-09-01'
              AND ServiceEndTime   >= '2026-09-01'
        ),
        redeem_poi AS (
            SELECT DISTINCT ow.OfferId
            FROM [Mars].[dbo].[OfferWallet] ow (NOLOCK)
            INNER JOIN [mars].[dbo].[VoucherOrder] vo (NOLOCK) ON vo.OfferId = ow.OfferId
            WHERE ow.RedeemPoiId IS NOT NULL
              AND vo.PaymentTime >= '2025-09-01' AND vo.PaymentTime < '2026-09-01'
              AND EXISTS (SELECT 1 FROM bookable_poi bp WHERE bp.poiid = ow.RedeemPoiId)
        ),
        menu_buyers_12m AS (
            SELECT DISTINCT b.userid
            FROM BookingMenuOrder bmo (NOLOCK)
            INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
            INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
            WHERE b.status = 10 AND bmo.status = 15
              AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
              AND bp.PaymentTime >= '2025-09-01' AND bp.PaymentTime < '2026-09-01'
              AND b.userid IS NOT NULL AND b.userid != 0
        )
        SELECT DISTINCT u.userid
        FROM VoucherOrder vo (NOLOCK)
        INNER JOIN redeem_poi rpl ON vo.OfferId = rpl.OfferId
        INNER JOIN [mars].[dbo].[User] u (NOLOCK) ON u.SSOUserId = vo.SSOUserId
        WHERE vo.PaymentTime >= '2025-09-01' AND vo.PaymentTime < '2026-09-01'
          AND vo.status = 10
          AND u.status = 10
          AND NOT EXISTS (SELECT 1 FROM menu_buyers_12m m WHERE m.userid = u.userid)
        ORDER BY u.userid
    """
}

# ── Gp8: Premium User ───────────────────────────────────────
groups['Gp8'] = {
    'sheet_name': 'Premium User',
    'promo':   '$1600-120',
    'sql': """
        USE Mars;
        SELECT DISTINCT b.userid
        FROM BookingMenuOrder bmo (NOLOCK)
        INNER JOIN BookingMenu bm (NOLOCK) ON bm.BookingMenuId = bmo.BookingMenuId
        INNER JOIN Booking b (NOLOCK) ON b.BookingId = bmo.BookingId
        INNER JOIN BookingPaymentTransaction bp (NOLOCK) ON bp.BookingId = b.BookingId
        INNER JOIN [Mars].[dbo].[User] u (NOLOCK) ON u.userid = b.userid
        WHERE b.status = 10 AND bmo.status = 15
          AND bp.PaymentStatus = 10 AND bp.BookingPaymentTransactionType = 1
          AND bm.BookingMenuType = 2
          AND bp.PaymentTime >= '2025-09-01' AND bp.PaymentTime < '2026-09-01'
          AND b.userid IS NOT NULL AND b.userid != 0
          AND u.status = 10
        ORDER BY b.userid
    """
}

print(f'{len(groups)} groups defined')
print(f'Sheet order: {[groups[g]["sheet_name"] for g in SHEET_ORDER]}')

8 groups defined
Sheet order: ['Sleeping User (A)', 'Warm User', 'New User(of previous mos)', 'Booking Standing 5 Star', 'Sleeping User (B)', 'NEW NEW User (A)', 'NEW NEW User (B)', 'Premium User']


## Step 2 — 查詢各 Group 並儲存 DataFrame

In [18]:
# ─── 逐個 Group 執行 SQL，讀返 DataFrame ─────────────────────
results = {}

for gp_name, info in groups.items():
    print(f'Querying {gp_name} ({info["sheet_name"]})...', end=' ')
    try:
        df = pd.read_sql(info['sql'], conn)
        df.columns = ['userid']
        results[gp_name] = df
        print(f'{len(df):,} rows')
    except Exception as e:
        print(f'ERROR: {e}')
        results[gp_name] = pd.DataFrame(columns=['userid'])

print('\nDone. Summary:')
for gp_name in SHEET_ORDER:
    df = results[gp_name]
    print(f'  {groups[gp_name]["sheet_name"]:30s} — {len(df):>6,} users')

Querying Gp1 (Sleeping User (A))... 

C:\Users\lenalee\AppData\Local\Temp\ipykernel_35472\263522019.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(info['sql'], conn)


46,213 rows
Querying Gp2 (Warm User)... 12,112 rows
Querying Gp3 (New User(of previous mos))... 6,469 rows
Querying Gp4 (Booking Standing 5 Star)... 13,661 rows
Querying Gp5 (Sleeping User (B))... 72,682 rows
Querying Gp6 (NEW NEW User (A))... 24,600 rows
Querying Gp7 (NEW NEW User (B))... 13,557 rows
Querying Gp8 (Premium User)... 10,519 rows

Done. Summary:
  Sleeping User (A)              — 46,213 users
  Warm User                      — 12,112 users
  New User(of previous mos)      —  6,469 users
  Booking Standing 5 Star        — 13,661 users
  Sleeping User (B)              — 72,682 users
  NEW NEW User (A)               — 24,600 users
  NEW NEW User (B)               — 13,557 users
  Premium User                   — 10,519 users


## Step 3 — Export 去 Excel（每個 Group 一個 Sheet，只有 userid）

In [19]:
# ─── Helper: 格式化 worksheet header (跟參考 Excel 格式) ─────
HEADER_FONT  = Font(bold=True)
HEADER_ALIGN = Alignment(horizontal='center', vertical='top')
COL_WIDTH    = 13.0


def style_sheet(ws):
    """Apply header formatting + fixed column width."""
    for cell in ws[1]:
        cell.font      = HEADER_FONT
        cell.alignment = HEADER_ALIGN
    ws.column_dimensions['A'].width = COL_WIDTH


# ─── Write Excel ─────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for gp_name in SHEET_ORDER:
        df = results[gp_name]
        sheet_name = groups[gp_name]['sheet_name']
        df[['userid']].to_excel(writer, sheet_name=sheet_name, index=False)
        style_sheet(writer.sheets[sheet_name])

print(f'Excel saved → {OUTPUT_PATH}')
print(f'Sheets: {[groups[g]["sheet_name"] for g in SHEET_ORDER]}')

Excel saved → Menu_Targeting_UserSegments.xlsx
Sheets: ['Sleeping User (A)', 'Warm User', 'New User(of previous mos)', 'Booking Standing 5 Star', 'Sleeping User (B)', 'NEW NEW User (A)', 'NEW NEW User (B)', 'Premium User']


## Step 4 — 驗證輸出

In [20]:
# ─── 讀返 Excel 驗證 ─────────────────────────────────────────
for gp_name in SHEET_ORDER:
    sheet_name = groups[gp_name]['sheet_name']
    df_sheet = pd.read_excel(OUTPUT_PATH, sheet_name=sheet_name)
    print(f'{sheet_name:30s} — {len(df_sheet):>6,} rows  (first 3: {df_sheet["userid"].head(3).tolist()})')

conn.close()
print('\nDB connection closed.')

Sleeping User (A)              — 46,213 rows  (first 3: [117, 150, 153])
Warm User                      — 12,112 rows  (first 3: [114, 209, 261])
New User(of previous mos)      —  6,469 rows  (first 3: [281, 447, 757])
Booking Standing 5 Star        — 13,661 rows  (first 3: [160, 232, 250])
Sleeping User (B)              — 72,682 rows  (first 3: [113, 203, 221])
NEW NEW User (A)               — 24,600 rows  (first 3: [180, 258, 284])
NEW NEW User (B)               — 13,557 rows  (first 3: [141, 370, 705])
Premium User                   — 10,519 rows  (first 3: [150, 153, 160])

DB connection closed.
